# Récupérer des images avec IIIF

L'[International Image Interoperability Framework](https://iiif.io/) est un standard de partage d'image en haute résolution sur le web. Il est utilisé par la majorité des institutions patrimoniales pour donner accès à leurs collections par une Interface (API) IIIF. Connaître le fonctionnement de IIIF permet d'automatiser le téléchargement de documents, et même de récupérer des informations comme des annotations liés aux images dans certains cas.

Le résultat d'une requếte IIIF prend la forme d'un fichier JSON relativement complexe. Voici plusieurs exemples de manifestes IIIF que vous pouvez explorer.

Cyclopaedia, 5e éd., Vol. 2 dont le  manifeste a été récupéré via la plateforme Aikon VHS.
```
https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/manifest.json
```
France, Tours, Bibliothèque diocésaine - Diocèse 08 dont le manifeste a été récupéré via la bibliotèque ARCA de l'IRHT.

```
https://api.irht.cnrs.fr/ark:/63955/fufe90re0j42/manifest.json
```

Un manifeste IIIF  est un document structuré qui décrit un objet numérique patrimonial en combinant des métadonnées descriptives et la liste ordonnée des images qui le composent (avec leurs identifiants, dimensions et zones de découpage).

# Interagir avec un manifeste IIIF

In [ ]:
import requests

In [ ]:
url = "https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/manifest.json" # Ajouter le lien manifeste IIIF

Les réponses les plus courantes sont :

- 200 : succès de la requête ;  
- 400 : accès refusé ;
- 500 : erreurs serveur ;

[Ressource de toutes les erreurs qu'un serveur peut vous renvoyer](https://http.cat/)

In [ ]:
resultat = requests.get(url)
resultat

<Response [200]>

In [ ]:
# Si la réponse de l'API est 403, nous devons alors faire croire au endpoint que nous sommes un utilisateur "humain" ou non automatique

headers = {
'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
'Accept': 'application/json'
}

resultat = requests.get(url, headers=headers)
resultat

<Response [200]>

In [ ]:
data = resultat.json()
data # permet de visualiser le résultat

{'@context': 'http://iiif.io/api/presentation/2/context.json',
 '@id': 'https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/manifest.json',
 '@type': 'sc:Manifest',
 'label': 'image #2562: Cyclopaedia, 5e éd., Vol. 2',
 'metadata': [{'label': 'Reference number', 'value': 'Volume n°2'},
  {'label': 'Work(s)', 'value': 'Ephraim Chambers | Cyclopaedia'},
  {'label': 'Place(s) of production', 'value': '-'},
  {'label': 'Dates', 'value': 1743},
  {'label': 'Document type', 'value': 'Typographie'},
  {'label': '@id', 'value': 'wit2320_img2562'}],
 'description': 'Computer Vision and Historical Analysis of Scientific Illustration Circulation',
 'attribution': 'VHS platform',
 'license': 'https://vhs.huma-num.fr/vhs/rgpd#license',
 'sequences': [{'@id': 'https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/sequence/normal.json',
   '@type': 'sc:Sequence',
   'label': 'Normal Order',
   'canvases': [{'@id': 'https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/canvas/c1.json',
     '@type': 'sc:Canvas',
 

In [ ]:
listeDesUrlDesImages = [] # Initialisation d'une liste vide

# Nous navigons ici dans le manifeste, les images sont stockées dans le sequences > canvases > images > ressources > @id
for canvas in data.get('sequences')[0].get('canvases'):
  images = canvas.get('images')
  for image in images:
    resource = image.get('resource')
    if resource.get('@id'):
      url = resource.get('@id')
      listeDesUrlDesImages.append(url)

listeDesUrlDesImages

['https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/full/full/0/default.jpg',
 'https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0002.jpg/full/full/0/default.jpg',
 'https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0003.jpg/full/full/0/default.jpg',
 'https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0004.jpg/full/full/0/default.jpg',
 'https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0005.jpg/full/full/0/default.jpg',
 'https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0006.jpg/full/full/0/default.jpg',
 'https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0007.jpg/full/full/0/default.jpg']

# Spécification d'une uri d'image IIIF

L'uri de requếte d'une image IIIF est toujours organisé de la même manière
```
{scheme}://{server}{/prefix}/{identifier}/{region}/{size}/{rotation}/{quality}.{format}
```
```
https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/full/full/0/default.jpg
```

Il est donc possible de faire passer des arguments directement via la requête IIIF.

Ici un carré au centre de l'image
```
https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/square/max/0/default.jpg
```

Ici un carré à l'envers
```
https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/square/max/180/default.jpg
```

# Rapatrier des images IIIF

In [ ]:
import time
import os

from tqdm import tqdm

os.makedirs('/content/imagesTéléchargés') # Nous créons ici un dossier dans l'environnement Jupyter pour y écrire nos images

In [ ]:
for index, url in enumerate(tqdm(listeDesUrlDesImages)): # Nous écrivons ici toutes les images contenues dans notre listeDesUrlImages
    imageData = requests.get(url).content
    with open(f'/content/imagesTéléchargés/{index}.jpg', 'wb') as image_file:
        image_file.write(imageData)
    time.sleep(2) # Nous laissons une pause de 2 secondes pour éviter de surcharger le serveur.

100%|██████████| 7/7 [00:19<00:00,  2.79s/it]


# Exemple d'utilisation de IIIF : Récupérer des annotations via Aikon

Cet exemple est un démonstrateur de l'utilité de pouvoir récupérer automatiquement des grands volumes d'images et de métadonnées associées avec IIIF.

Il se base sur cet [application](https://github.com/Anarchiviste/aikoncli/blob/main/aikoncli.py)

In [39]:
from pathlib import Path

url_manifest = "https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/manifest.json"
output_dir = "IIIFToYolo"

output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

In [40]:
import requests
import re
import json

from pathlib import Path
from tqdm import tqdm

def iiif_url_to_annotation_url(url_manifest: str) -> str:
    """
    Converts a IIIF manifest URL to its corresponding annotation URL.

    :param url_manifest: The IIIF manifest URL.
    :return: The annotation URL derived from the manifest URL.
    :raises ValueError: If the URL does not match the expected pattern.
    """
    match = re.search(r"(^https:\/\/.*)\/(.*)\/(.*\/wit)(\d+)(.*)\/(manifest\.json)", url_manifest)

    if match:
        radical_url = match.group(1)
        instance = match.group(2)
        witness_id = match.group(4)
        url_annotations = f"{radical_url}/{instance}/witness/{witness_id}/regions/canvas"
    else:
        raise ValueError('Invalide link')

    return url_annotations

def downloading_image_from_manifest(url_manifest: str, output_dir: Path) -> None:
    """
    Downloads all images from a IIIF manifest and saves them locally.
    Also builds a dictionary mapping image IDs to their metadata (url, height, width).

    :param url_manifest: The IIIF manifest URL.
    :param output_dir: The directory where images will be saved.
    :return: A tuple containing the image metadata dictionary and the witness name.
    """
    request = requests.get(url_manifest)
    manifest = request.json()
    metadata = manifest['metadata']

    meta = {item['label']: item['value'] for item in metadata}
    name = meta['@id'].replace(' ', '')
    images_dir = Path(f"{output_dir}/{name}/images")
    images_dir.mkdir(parents=True, exist_ok=True)

    canvases = manifest['sequences'][0]['canvases']
    i_id = 0
    img_dict = {}
    for canvas in tqdm(canvases, desc="Downloading"):
        for image in canvas['images']:
            resource = image['resource']
            i_id += 1
            img_dict[i_id] = {"url": resource['@id'], "height": resource['height'], "width": resource['width']}
            data = requests.get(resource['@id']).content
            with open(f'{images_dir}/{i_id}.jpg', 'wb') as f:
                f.write(data)

    return img_dict, name

def iiif_annotations_to_yolo(img_dict: dict, url_annotations: str, output_dir: Path, name: str) -> None:
    """
    Fetches annotations from an AIKON API endpoint and converts them to YOLO format label files.
    Each annotation's bounding box is normalized relative to its image dimensions.

    :param img_dict: Dictionary mapping image IDs to their metadata (url, height, width).
    :param url_annotations: The URL of the annotation endpoint.
    :param output_dir: The directory where label files will be saved.
    :param name: The witness name, used to build the output directory path.
    """
    labels_dir = Path(f"{output_dir}/{name}/labels")
    labels_dir.mkdir(parents=True, exist_ok=True)
    resultat_annotations = requests.get(url_annotations)
    print("Status :", resultat_annotations.status_code)
    print("Contenu :", repr(resultat_annotations.text[:300]))
    annotations_manifest = resultat_annotations.json()

    i_index = []

    for i in annotations_manifest:
        annotations = annotations_manifest.get(i)
        page_dictionnaire = img_dict.get(int(i))
        img_height = page_dictionnaire.get('height')
        img_width = page_dictionnaire.get('width')
        lines = {}

        for annotation in annotations:
            img_id = (annotations.get(annotation).get('canvas'))
            x_tl, y_tl, w, h = annotations.get(annotation).get('xywh')
            x_center = (x_tl + w / 2) / img_width
            y_center = (y_tl + h / 2) / img_height
            width = w / img_width
            height = h / img_height

            lines.setdefault(img_id, []).append(f'0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')

        for img_id, img_lines in lines.items():
            with open(f'{labels_dir}/{img_id}.txt', 'w') as f:
                f.write("\n".join(img_lines))

def aikon_to_yolo(url_manifest: str, output_dir: str):
    """
    Orchestrates the full pipeline: converts a IIIF manifest URL to an annotation URL,
    downloads all images, and converts annotations to YOLO format label files.

    :param url_manifest: The IIIF manifest URL.
    :param output_dir: The directory where all outputs will be saved.
    """
    url_annotations = iiif_url_to_annotation_url(url_manifest)
    img_dict, name = downloading_image_from_manifest(url_manifest, output_dir)
    iiif_annotations_to_yolo(img_dict, url_annotations, output_dir, name)

aikon_to_yolo(url_manifest, output_dir)

Downloading: 100%|██████████| 7/7 [00:05<00:00,  1.31it/s]


Status : 200
Contenu : '{"1": {}, "2": {"c2_0422db54-00fb-4e61-a954-72f851c9b253": {"id": "c2_0422db54-00fb-4e61-a954-72f851c9b253", "ref": "wit2320_img2562_0002_958,1264,160,155", "class": "Region", "type": "Extraction de r\\u00e9gions", "title": "Canvas 2 \\u2013 958,1264,160,155", "url": "https://vhs.huma-num.fr/iiif/2/wi'
